In [ ]:
# Setting up imports and loading the cohort.
import sys
sys.path.append("..")
import json
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Splitting the cohort into failed and live companies.
cohort = pd.read_parquet("../data/cohort.parquet")
failed = cohort[cohort["is_failed"] == 1]
live = cohort[cohort["is_failed"] == 0]
print(len(failed), len(live))

1500 1500


In [ ]:
# Reloading the failure date function from Week 3.
def get_failure_date(filings_items):
    """Return the date liquidation started, or None if no clear marker exists."""
    for target_type in ("600", "COCOMP"):
        matches = [item["date"] for item in filings_items if item.get("type") == target_type]
        if matches:
            return min(matches)
    return None

In [ ]:
# Reloading the leakage filter from Week 3.
def filter_before_snapshot(data: dict, snapshot_date: pd.Timestamp) -> dict:
    """Return a version of a company's data containing only information
    available before its snapshot date.
    """
    filings = [
        item for item in data["filings"]["items"]
        if pd.to_datetime(item.get("date")) < snapshot_date
    ]
    officers = [
        o for o in data["officers"]["items"]
        if pd.to_datetime(o.get("appointed_on", "1900-01-01")) < snapshot_date
    ]
    return {"profile": data["profile"], "filings": filings, "officers": officers}

In [ ]:
# Computing the failure date and snapshot date for each failed company.
failure_dates = {}
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    date = get_failure_date(data["filings"]["items"])
    if date:
        failure_dates[number] = date

len(failure_dates)

1253

In [ ]:
# Building a table of failure dates and snapshot dates, 12 months before failure.
failure_df = pd.DataFrame({
    "CompanyNumber": list(failure_dates.keys()),
    "failure_date": pd.to_datetime(list(failure_dates.values())),
})
failure_df["snapshot_date"] = failure_df["failure_date"] - pd.DateOffset(months=12)

failure_df.head()

,CompanyNumber,failure_date,snapshot_date
0,09977882,2024-11-26,2023-11-26
1,04544271,2022-10-31,2021-10-31
2,01287947,2025-10-08,2024-10-08
3,07161144,2026-06-26,2025-06-26
4,07682840,2022-12-17,2021-12-17


In [ ]:
# Converting incorporation dates to real dates so we can compare them to snapshot dates.
live = live.copy()
live["IncorporationDate"] = pd.to_datetime(live["IncorporationDate"], format="%d/%m/%Y")

In [ ]:
# Assigning each live company a snapshot date sampled from the failed companies snapshot dates,
# constrained to dates after that company's own incorporation.
rng = np.random.default_rng(1)
snapshot_pool = failure_df["snapshot_date"].to_numpy()

def sample_valid_snapshot(incorporation_date):
    valid = snapshot_pool[snapshot_pool > np.datetime64(incorporation_date)]
    if len(valid) == 0:
        return pd.NaT
    return rng.choice(valid)

live["snapshot_date"] = live["IncorporationDate"].apply(sample_valid_snapshot)

In [ ]:
# Dropping live companies that could not get a valid snapshot date, and checking the count.
live_final = live.dropna(subset=["snapshot_date"]).copy()
len(live_final)

1266

In [13]:
# Feature 1: days overdue on accounts, as of the snapshot date.
def days_overdue_on_accounts(profile: dict, snapshot_date: pd.Timestamp) -> float:
    """Return how many days overdue the company's accounts were, as of the
    snapshot date. Negative means not yet due. Missing due date returns NaN.
    """
    accounts = profile.get("accounts", {})
    due_on = accounts.get("next_accounts", {}).get("due_on")
    if not due_on:
        return float("nan")
    due_date = pd.to_datetime(due_on)
    return (snapshot_date - due_date).days

In [14]:
# Test the feature on one failed company.
test_number = failure_df["CompanyNumber"].iloc[0]
test_snapshot = failure_df.loc[failure_df["CompanyNumber"] == test_number, "snapshot_date"].iloc[0]
test_data = json.loads((Path("../data/raw") / f"{test_number}.json").read_text())

days_overdue_on_accounts(test_data["profile"], test_snapshot)

-491

In [16]:
# Check whether filing descriptions ever mention lateness directly.
sample_descriptions = set()
for item in test_data["filings"]["items"]:
    sample_descriptions.add(item.get("description", ""))

sample_descriptions

{'accounts-amended-with-accounts-type-total-exemption-full',
 'accounts-with-accounts-type-micro-entity',
 'accounts-with-accounts-type-total-exemption-full',
 'change-account-reference-date-company-previous-extended',
 'change-registered-office-address-company-with-date-old-address-new-address',
 'confirmation-statement-with-no-updates',
 'confirmation-statement-with-updates',
 'gazette-filings-brought-up-to-date',
 'gazette-notice-compulsory',
 'liquidation-disclaimer-notice',
 'liquidation-voluntary-appointment-of-liquidator',
 'liquidation-voluntary-statement-of-affairs',
 'liquidation-voluntary-statement-of-receipts-and-payments-with-brought-down-date',
 'mortgage-create-with-deed-with-charge-number-charge-creation-date',
 'mortgage-satisfy-charge-full',
 'resolution'}

In [17]:
# Look at the gaps between consecutive confirmation statement filings for one company.
confirmation_filings = sorted(
    [f["date"] for f in test_data["filings"]["items"] if f.get("type") == "CS01"]
)
confirmation_filings

['2020-03-31', '2021-04-19', '2022-02-11', '2022-11-30', '2023-10-24']

In [18]:
# Calculate the number of days between each consecutive confirmation statement.
dates = pd.to_datetime(confirmation_filings)
gaps = dates.to_series().diff().dropna()
gaps.dt.days

2021-04-19    384
2022-02-11    298
2022-11-30    292
2023-10-24    328
dtype: int64

In [19]:
# Feature 2: count of late confirmation statements in the filing history before the snapshot.
def count_late_confirmation_statements(filings: list) -> int:
    """Count confirmation statements filed more than 379 days after the
    previous one (12 months plus the 14 day statutory grace period).
    """
    dates = sorted(pd.to_datetime(f["date"]) for f in filings if f.get("type") == "CS01")
    if len(dates) < 2:
        return 0
    gaps = pd.Series(dates).diff().dropna().dt.days
    return int((gaps > 379).sum())

In [20]:
# Test the feature on the same example company.
count_late_confirmation_statements(test_data["filings"]["items"])

1

In [21]:
# Look at the structure of one officer record to see what fields are available for resignations.
test_data["officers"]["items"][0]

{'etag': 'c1e0e07d78ac5897e21e3fbd508f67b145fa6c0a',
 'address': {'address_line_1': 'Riverside 2, No.3, Campbell Road',
  'locality': 'Stoke On Trent',
  'postal_code': 'ST4 4RJ',
  'premises': 'C/O Currie Young Limited'},
 'appointed_on': '2016-01-29',
 'is_pre_1992_appointment': False,
 'country_of_residence': 'England',
 'date_of_birth': {'month': 9, 'year': 1971},
 'links': {'self': '/company/09977882/appointments/3vzn5oq3UYhx3PpUZ1DmK4CiAuM',
  'officer': {'appointments': '/officers/dBbjyr3J-1zlEohGF0H_G7DGesk/appointments'}},
 'name': 'LOGAN, Mara Maranda Mellissa',
 'nationality': 'British',
 'officer_role': 'director',
 'person_number': '204661170001',
 'identity_verification_details': {'appointment_verification_statement_due_on': '2025-11-18'}}

In [22]:
# Find an officer record that includes a resignation date, to confirm the field name.
for o in test_data["officers"]["items"]:
    if "resigned_on" in o:
        print(o["name"], o["resigned_on"])

TAYLOR, Jade Clairnese Minnet 2020-03-20
TAYLOR, Jade Clairnese Minnet 2018-02-14
TAYLOR, Keiran Martin 2019-04-16


In [23]:
# Feature 3: count of director resignations in the 12 months before the snapshot date.
def count_recent_resignations(officers: list, snapshot_date: pd.Timestamp) -> int:
    """Count officers who resigned in the 12 months before the snapshot date."""
    window_start = snapshot_date - pd.DateOffset(months=12)
    count = 0
    for o in officers:
        resigned_on = o.get("resigned_on")
        if resigned_on and window_start <= pd.to_datetime(resigned_on) < snapshot_date:
            count += 1
    return count

In [24]:
# Test the feature on the same example company.
count_recent_resignations(test_data["officers"]["items"], test_snapshot)

0

In [25]:
# Look at how charges are recorded in the filing history, to find the right filing type.
charge_related = set()
for item in test_data["filings"]["items"]:
    if "MR" in item.get("type", "") or "mortgage" in item.get("description", "").lower():
        charge_related.add((item["type"], item["description"]))

charge_related

{('MR01', 'mortgage-create-with-deed-with-charge-number-charge-creation-date'),
 ('MR04', 'mortgage-satisfy-charge-full')}

In [26]:
# Feature 4: count of new charges registered in the 24 months before the snapshot date.
def count_new_charges(filings: list, snapshot_date: pd.Timestamp) -> int:
    """Count MR01 filings (new charge created) in the 24 months before the
    snapshot date. A charge is a lender securing debt against company assets.
    """
    window_start = snapshot_date - pd.DateOffset(months=24)
    count = 0
    for f in filings:
        if f.get("type") == "MR01":
            filed_on = pd.to_datetime(f["date"])
            if window_start <= filed_on < snapshot_date:
                count += 1
    return count

In [27]:
# Test the feature on the same example company.
count_new_charges(test_data["filings"]["items"], test_snapshot)

1

In [28]:
# Feature 5: company age in years, as of the snapshot date.
def company_age_years(profile: dict, snapshot_date: pd.Timestamp) -> float:
    """Return the company's age in years at the snapshot date."""
    incorporated = pd.to_datetime(profile["date_of_creation"])
    return (snapshot_date - incorporated).days / 365.25

In [29]:
# Test the feature on the same example company.
company_age_years(test_data["profile"], test_snapshot)

7.824777549623546

In [30]:
# Feature 6: longest gap in days between any two consecutive filings before the snapshot.
def longest_filing_gap(filings: list, snapshot_date: pd.Timestamp) -> float:
    """Return the longest gap in days between consecutive filings before the
    snapshot date. A long gap can indicate a company has gone quiet.
    """
    dates = sorted(pd.to_datetime(f["date"]) for f in filings if pd.to_datetime(f["date"]) < snapshot_date)
    if len(dates) < 2:
        return float("nan")
    gaps = pd.Series(dates).diff().dropna().dt.days
    return float(gaps.max())

In [31]:
# Test the feature on the same example company.
longest_filing_gap(test_data["filings"]["items"], test_snapshot)

287.0

In [32]:
# Build the full features table for the failed companies.
failed_rows = []
for _, row in failure_df.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    failed_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_overdue_on_accounts": days_overdue_on_accounts(filtered["profile"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 1,
    })

len(failed_rows)

1253

In [33]:
# Build the full features table for the live companies.
live_rows = []
for _, row in live_final.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    live_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_overdue_on_accounts": days_overdue_on_accounts(filtered["profile"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 0,
    })

len(live_rows)

1266

In [34]:
# Combine failed and live features into one table.
features_df = pd.DataFrame(failed_rows + live_rows)
features_df.shape

(2519, 9)

In [35]:
# Check missing values across all features.
features_df.isna().sum()

CompanyNumber                           0
snapshot_date                           0
days_overdue_on_accounts               84
count_late_confirmation_statements      0
count_recent_resignations               0
count_new_charges                       0
company_age_years                       0
longest_filing_gap                    391
is_failed                               0
dtype: int64

In [36]:
# Check whether the missing days_overdue_on_accounts cases are young companies or something else.
missing_overdue = features_df[features_df["days_overdue_on_accounts"].isna()]
missing_overdue["company_age_years"].describe()

count     84.000000
mean       9.854829
std       13.600465
min       -0.443532
25%        2.643395
50%        5.902806
75%       11.273101
max      103.868583
Name: company_age_years, dtype: float64

In [37]:
# Investigate the negative company age - this should not be possible.
features_df[features_df["company_age_years"] < 0]

,CompanyNumber,snapshot_date,days_overdue_on_accounts,count_late_confirmation_statements,count_recent_resignations,count_new_charges,company_age_years,longest_filing_gap,is_failed
19,15532142,2023-09-20,NaN,0,0,0,-0.443532,NaN,1
327,16311665,2025-02-11,-669.0,0,0,0,-0.079398,NaN,1
383,16256180,2024-12-18,-699.0,0,0,0,-0.167009,NaN,1
499,09781534,2015-08-30,-657.0,0,0,0,-0.049281,NaN,1
569,05239355,2004-05-03,-811.0,0,0,0,-0.391513,NaN,1
752,14496808,2022-11-14,-646.0,0,0,0,-0.019165,NaN,1
837,11978556,2018-11-23,-892.0,0,0,0,-0.440794,NaN,1
1172,15222119,2023-07-26,-724.0,0,0,0,-0.232717,NaN,1


In [38]:
# Exclude failed companies whose snapshot date falls before their incorporation (failed within 12 months).
features_df = features_df[features_df["company_age_years"] >= 0].copy()
features_df.shape

(2511, 9)

In [39]:
# Check the account category for companies missing days_overdue_on_accounts.
missing_overdue_numbers = features_df[features_df["days_overdue_on_accounts"].isna()]["CompanyNumber"]
categories = []
for number in missing_overdue_numbers:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    categories.append(data["profile"].get("accounts", {}).get("last_accounts", {}).get("type", "none"))

pd.Series(categories).value_counts()

full                     21
small                    16
null                     14
micro-entity             12
total-exemption-full      9
dormant                   6
unaudited-abridged        2
audited-abridged          1
none                      1
total-exemption-small     1
Name: count, dtype: int64

In [40]:
# Checking the 'can_file' flag for companies missing days_overdue_on_accounts.
can_file_values = []
for number in missing_overdue_numbers:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    can_file_values.append(data["profile"].get("can_file"))

pd.Series(can_file_values).value_counts()

True     52
False    31
Name: count, dtype: int64

In [41]:
# Looking directly at the accounts object for a few companies missing days_overdue_on_accounts.
for number in list(missing_overdue_numbers)[:5]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    print(number, data["profile"].get("accounts"))

01141767 {'accounting_reference_date': {'day': '31', 'month': '01'}, 'last_accounts': {'made_up_to': '2024-01-31', 'period_end_on': '2024-01-31', 'type': 'total-exemption-full'}}
02247011 {'accounting_reference_date': {'day': '31', 'month': '07'}, 'last_accounts': {'type': 'null'}, 'next_due': '1992-05-31', 'next_made_up_to': '1991-07-31', 'overdue': True}
02480041 {'accounting_reference_date': {'day': '31', 'month': '03'}, 'last_accounts': {'type': 'null'}, 'next_due': '1992-01-12', 'next_made_up_to': '1991-03-31', 'overdue': True}
05351394 {'accounting_reference_date': {'day': '31', 'month': '12'}, 'last_accounts': {'made_up_to': '2023-12-31', 'period_end_on': '2023-12-31', 'type': 'full'}}
02641955 {'accounting_reference_date': {'day': '31', 'month': '10'}, 'last_accounts': {'made_up_to': '1993-10-31', 'type': 'full'}, 'next_due': '1995-08-31', 'next_made_up_to': '1994-10-31', 'overdue': True}


In [42]:
# Fixing days_overdue_on_accounts to also check the older top level next_due field.
def days_overdue_on_accounts(profile: dict, snapshot_date: pd.Timestamp) -> float:
    """Returning how many days overdue the company's accounts were, as of the
    snapshot date. Falls back to the older top level next_due field for
    accounts records that predate the newer next_accounts structure.
    """
    accounts = profile.get("accounts", {})
    due_on = accounts.get("next_accounts", {}).get("due_on") or accounts.get("next_due")
    if not due_on:
        return float("nan")
    due_date = pd.to_datetime(due_on)
    return (snapshot_date - due_date).days

In [43]:
# Rechecking missing values after fixing the accounts due date fallback.
features_df.isna().sum()

CompanyNumber                           0
snapshot_date                           0
days_overdue_on_accounts               83
count_late_confirmation_statements      0
count_recent_resignations               0
count_new_charges                       0
company_age_years                       0
longest_filing_gap                    383
is_failed                               0
dtype: int64

In [44]:
# Adding missingness flags before filling, so the model can learn from absence itself.
features_df["days_overdue_missing"] = features_df["days_overdue_on_accounts"].isna().astype(int)
features_df["filing_gap_missing"] = features_df["longest_filing_gap"].isna().astype(int)

features_df["days_overdue_on_accounts"] = features_df["days_overdue_on_accounts"].fillna(0)
features_df["longest_filing_gap"] = features_df["longest_filing_gap"].fillna(0)

features_df.isna().sum()

CompanyNumber                         0
snapshot_date                         0
days_overdue_on_accounts              0
count_late_confirmation_statements    0
count_recent_resignations             0
count_new_charges                     0
company_age_years                     0
longest_filing_gap                    0
is_failed                             0
days_overdue_missing                  0
filing_gap_missing                    0
dtype: int64

In [45]:
# Saving the final features table for modelling.
features_df.to_parquet("../data/features.parquet", index=False)
features_df.shape

(2511, 11)

In [46]:
# Splitting into train and test by snapshot date, not randomly, since this respects time order.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [47]:
# Building the dumb baseline: flag anyone with overdue accounts, and scoring it on the test set.
from sklearn.metrics import precision_score, recall_score

baseline_pred = (test["days_overdue_on_accounts"] > 0).astype(int)

baseline_precision = precision_score(test["is_failed"], baseline_pred)
baseline_recall = recall_score(test["is_failed"], baseline_pred)

print(baseline_precision, baseline_recall)

1.0 0.062111801242236024


In [48]:
# Defining a precision-at-top-k metric, since that matches how this model would actually be used.
def precision_at_k(y_true, y_scores, k_percent=10):
    """Return the precision among the top k percent highest scored companies."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    return y_true.iloc[top_k_idx].mean()

In [49]:
# Checking baseline precision at top 10% using days overdue as the ranking score.
precision_at_k(test["is_failed"].reset_index(drop=True), test["days_overdue_on_accounts"].reset_index(drop=True), k_percent=10)

np.float64(0.98)

In [74]:
!pip install xgboost

In [53]:
# Preparing the feature columns and training XGBoost on the time based train split.
import xgboost as xgb

feature_cols = [
    "days_overdue_on_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "days_overdue_missing", "filing_gap_missing",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [54]:
# Scoring the model on the test set using precision at top 10 percent, for a fair comparison.
model_scores = model.predict_proba(test[feature_cols])[:, 1]
precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores), k_percent=10)

np.float64(0.94)

In [55]:
# Comparing how many of the total test set failures each ranking actually caught in its top 10 percent.
def recall_at_k(y_true, y_scores, k_percent=10):
    """Return the share of all actual failures captured within the top k percent by score."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    caught = y_true.iloc[top_k_idx].sum()
    total_failures = y_true.sum()
    return caught / total_failures

baseline_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), test["days_overdue_on_accounts"].reset_index(drop=True))
model_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))

print("baseline recall at top 10%:", baseline_recall_at_10)
print("model recall at top 10%:", model_recall_at_10)

baseline recall at top 10%: 0.30434782608695654
model recall at top 10%: 0.2919254658385093


In [56]:
# Checking how much real signal exists in each feature - are some almost always zero?
train[feature_cols].describe()

,days_overdue_on_accounts,count_late_confirmation_statements,count_recent_resignations,count_new_charges,company_age_years,longest_filing_gap,days_overdue_missing,filing_gap_missing
count,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000
mean,-857.688247,0.360060,0.172809,0.073207,10.460777,277.067231,0.041335,0.119522
std,1086.405865,0.637898,0.550868,0.358705,12.926053,156.517227,0.199113,0.324482
min,-11134.000000,0.000000,0.000000,0.000000,0.005476,0.000000,0.000000,0.000000
25%,-1109.000000,0.000000,0.000000,0.000000,2.943874,228.750000,0.000000,0.000000
50%,-688.500000,0.000000,0.000000,0.000000,6.461328,310.000000,0.000000,0.000000
75%,-389.750000,1.000000,0.000000,0.000000,12.561259,368.000000,0.000000,0.000000
max,4956.000000,4.000000,6.000000,4.000000,146.715948,3500.000000,1.000000,1.000000


In [57]:
# Investigating the extreme negative outlier in days_overdue_on_accounts.
train[train["days_overdue_on_accounts"] < -5000][["CompanyNumber", "days_overdue_on_accounts", "is_failed"]]

,CompanyNumber,days_overdue_on_accounts,is_failed
1628,02167764,-10208.0,0
1579,01660626,-11134.0,0
2180,03286888,-10158.0,0
2294,02743240,-9783.0,0
2199,01223909,-9783.0,0
1927,03682947,-8940.0,0
1800,03152388,-7412.0,0
1708,04808548,-7324.0,0
1856,04086905,-6840.0,0
2428,03839960,-6493.0,0


In [58]:
# Looking directly at the accounts object for one extreme outlier company.
outlier_number = "01660626"
outlier_data = json.loads((Path("../data/raw") / f"{outlier_number}.json").read_text())
outlier_data["profile"].get("accounts")

{'accounting_reference_date': {'day': '31', 'month': '01'},
 'last_accounts': {'made_up_to': '2026-01-31',
  'period_end_on': '2026-01-31',
  'period_start_on': '2025-02-01',
  'type': 'total-exemption-full'},
 'next_accounts': {'due_on': '2027-10-31',
  'overdue': False,
  'period_end_on': '2027-01-31',
  'period_start_on': '2026-02-01'},
 'next_due': '2027-10-31',
 'next_made_up_to': '2027-01-31',
 'overdue': False}

In [59]:
# Checking accounts related filing types in one company's filtered history.
accounts_types = set()
for item in test_data["filings"]["items"]:
    t = item.get("type", "")
    if t.startswith("AA"):
        accounts_types.add((t, item.get("description", "")))

accounts_types

{('AA', 'accounts-with-accounts-type-micro-entity'),
 ('AA', 'accounts-with-accounts-type-total-exemption-full'),
 ('AA01', 'change-account-reference-date-company-previous-extended'),
 ('AAMD', 'accounts-amended-with-accounts-type-total-exemption-full')}

In [60]:
# Rebuilding days_overdue_on_accounts using only filing history, so it works correctly
# for live companies with older snapshot dates too, not just the current live profile.
def days_since_last_accounts(filings: list, snapshot_date: pd.Timestamp) -> float:
    """Return days since the most recent accounts filing before the snapshot
    date. A long gap suggests the company has fallen behind on statutory
    filing. Missing entirely (never filed) returns NaN.
    """
    accounts_dates = [
        pd.to_datetime(f["date"]) for f in filings if f.get("type") in ("AA", "AAMD")
    ]
    if not accounts_dates:
        return float("nan")
    return (snapshot_date - max(accounts_dates)).days

In [63]:
# Testing the rebuilt feature on the same example company.
filtered_test = filter_before_snapshot(test_data, test_snapshot)
days_since_last_accounts(filtered_test["filings"], test_snapshot)

382

In [64]:
# Rebuilding the features table for failed companies using the corrected accounts feature.
failed_rows = []
for _, row in failure_df.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    failed_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_since_last_accounts": days_since_last_accounts(filtered["filings"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 1,
    })

len(failed_rows)

1253

In [65]:
# Rebuilding the features table for live companies using the corrected accounts feature.
live_rows = []
for _, row in live_final.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    filtered = filter_before_snapshot(data, snapshot)

    live_rows.append({
        "CompanyNumber": number,
        "snapshot_date": snapshot,
        "days_since_last_accounts": days_since_last_accounts(filtered["filings"], snapshot),
        "count_late_confirmation_statements": count_late_confirmation_statements(filtered["filings"]),
        "count_recent_resignations": count_recent_resignations(filtered["officers"], snapshot),
        "count_new_charges": count_new_charges(filtered["filings"], snapshot),
        "company_age_years": company_age_years(filtered["profile"], snapshot),
        "longest_filing_gap": longest_filing_gap(filtered["filings"], snapshot),
        "is_failed": 0,
    })

len(live_rows)

1266

In [66]:
# Combining failed and live features into one table, with the corrected accounts feature.
features_df = pd.DataFrame(failed_rows + live_rows)
features_df.shape

(2519, 9)

In [67]:
# Re-excluding failed companies whose snapshot date falls before their incorporation.
features_df = features_df[features_df["company_age_years"] >= 0].copy()
features_df.shape

(2511, 9)

In [68]:
# Checking missing values with the corrected accounts feature.
features_df.isna().sum()

CompanyNumber                           0
snapshot_date                           0
days_since_last_accounts              639
count_late_confirmation_statements      0
count_recent_resignations               0
count_new_charges                       0
company_age_years                       0
longest_filing_gap                    383
is_failed                               0
dtype: int64

In [69]:
# Adding missingness flags before filling, so the model can learn from absence itself.
features_df["accounts_missing"] = features_df["days_since_last_accounts"].isna().astype(int)
features_df["filing_gap_missing"] = features_df["longest_filing_gap"].isna().astype(int)

features_df["days_since_last_accounts"] = features_df["days_since_last_accounts"].fillna(0)
features_df["longest_filing_gap"] = features_df["longest_filing_gap"].fillna(0)

features_df.isna().sum()

CompanyNumber                         0
snapshot_date                         0
days_since_last_accounts              0
count_late_confirmation_statements    0
count_recent_resignations             0
count_new_charges                     0
company_age_years                     0
longest_filing_gap                    0
is_failed                             0
accounts_missing                      0
filing_gap_missing                    0
dtype: int64